# Chapter 6 — Medallion Architecture Pipeline
**Catalog:** `dev` | **Schema:** `dbx_course`

This notebook builds a complete Bronze → Silver → Gold pipeline on Databricks using Delta Lake and Unity Catalog.
Each section corresponds to one lecture.
* You can run this notebook in Databricks free edition.
* Make sure the catalog and schema are already created.

---
## Section 1: Bronze Layer
**Design principle:** 
Bronze has one job: preserve what arrived. No more, no less.

In production, the Bronze layer is the contract between your pipeline and the outside world.\
Everything upstream of Bronze is someone else's system — a source database, an API, a file drop on S3. You do not control it. It can change schema without warning. It can send duplicates. It can send nulls in columns that should never be null. Bronze absorbs all of that without complaint.

The design rules for Bronze are strict precisely because of this: store everything as-is, add source metadata, never transform, never filter, never reject. If you break any of those rules, you have made Bronze into something it is not — and you have lost your audit trail in the process.

In [0]:
%python
# Inline source data — simulates daily CSV files arriving from S3
# In production these arrive via Lakeflow Connect or a scheduled copy job

orders_csv = """order_id,customer_id,product_id,quantity,unit_price,order_date,status
1001,C001,P01,2,29.99,2024-01-15,completed
1002,C002,P02,1,149.99,2024-01-15,completed
1003,C001,P03,3,9.99,2024-01-16,completed
1004,C003,P01,1,29.99,2024-01-16,pending
1005,C002,P02,2,149.99,2024-01-17,completed
1006,C004,P04,1,199.99,2024-01-17,cancelled
1007,C001,P01,4,29.99,2024-01-18,completed
1008,C003,P03,2,9.99,2024-01-18,completed
1001,C001,P01,2,29.99,2024-01-15,completed"""

#The orders feed has a duplicate — order 1001 appears twice. 

orders_header_row =[row.split(',') for row in orders_csv.strip().split('\n')[:1]][0]
orders_data_rows = [row.split(',') for row in orders_csv.strip().split('\n')[1:]]

customers_csv = """customer_id,first_name,last_name,email,city,signup_date,tier
C001,Aisha,Patel,aisha.patel@email.com,New York,2023-03-10,gold
C002,Marcus,Chen,marcus.chen@email.com,San Francisco,2023-05-22,silver
C003,Priya,Nair,priya.nair@email.com,Chicago,2023-07-14,bronze
C004,James,Okafor,james.okafor@email.com,Austin,2023-11-30,bronze
C001,Aisha,Patel,aisha.patel@email.com,New York,2023-03-10,gold"""

customers_header_row =[row.split(',') for row in customers_csv.strip().split('\n')[:1]][0]
customers_data_rows = [row.split(',') for row in customers_csv.strip().split('\n')[1:]]

# The customers feed has a duplicate too — C001 appears twice. 
# This is intentional. Real source systems send duplicates. Bronze accepts them. We will deal with them in Silver.


orders_df = spark.createDataFrame(orders_data_rows, orders_header_row)
customers_df = spark.createDataFrame(customers_data_rows, customers_header_row)

print(f"Orders rows: {orders_df.count()}")
print(f"Customers rows: {customers_df.count()}")

In [0]:
%python
# Add pipeline metadata columns — these make Bronze an audit trail
# _ingest_timestamp: when did this batch arrive
# _source: which feed produced this record
# _source_file: which file specifically (important for debugging)
# Underscore prefix is convention: pipeline-generated columns, not source data

from pyspark.sql import functions as F
import datetime

ingest_ts = F.lit(datetime.datetime(2024, 1, 19, 8, 0, 0))

orders_bronze = (
    orders_df
    .withColumn("_ingest_timestamp", ingest_ts)
    .withColumn("_source", F.lit("orders_feed"))
    .withColumn("_source_file", F.lit("orders_20240119.csv"))
)

customers_bronze = (
    customers_df
    .withColumn("_ingest_timestamp", ingest_ts)
    .withColumn("_source", F.lit("customers_feed"))
    .withColumn("_source_file", F.lit("customers_20240119.csv"))
)

In [0]:
%sql
-- Drop for clean demo run
DROP TABLE IF EXISTS dev.dbx_course.bronze_orders;
DROP TABLE IF EXISTS dev.dbx_course.bronze_customers;

In [0]:
%python
# Write Bronze tables
# Production note: use mode("append") for incremental daily loads
# overwrite is used here only to support clean demo re-runs

(
    orders_bronze.write
    .format("delta")
    .mode("overwrite") #In production, your Bronze write mode is almost never `overwrite`. It is `append`.
    .saveAsTable("dev.dbx_course.bronze_orders")
)

(
    customers_bronze.write
    .format("delta")
    .mode("overwrite") #In production, your Bronze write mode is almost never `overwrite`. It is `append`.
    .saveAsTable("dev.dbx_course.bronze_customers")
)

print("Bronze tables written.")

In [0]:
%sql
-- Verify Bronze orders — duplicate 1001 should appear twice
SELECT * FROM dev.dbx_course.bronze_orders ORDER BY order_id;

In [0]:
%sql
-- Verify Bronze customers — duplicate C001 should appear twice
SELECT * FROM dev.dbx_course.bronze_customers ORDER BY customer_id;

In [0]:
%sql
-- Confirm metadata columns are present
DESCRIBE TABLE dev.dbx_course.bronze_orders;

---
## Section 2: Silver Layer
**Design principle:** Enforce contracts.\
Bronze accepted everything. Silver enforces contracts. Every record that enters Silver has been cleaned, typed, deduplicated, and validated. Every record that leaves Silver meets a quality bar your downstream consumers can rely on.

This is the layer your data scientists query for exploration. This is the layer your Gold aggregations read from. If Silver is wrong, everything built on top of it is wrong. The production cost of a bad Silver layer is not a broken pipeline — it is wrong numbers in a dashboard that no one catches for many months.

The two most important jobs Silver does are deduplication and type enforcement. 
* Deduplication because sources send duplicates and downstream consumers cannot handle them. 
* Type enforcement because everything in Bronze is a string — unit prices, dates, quantities — and you cannot aggregate strings.

In [0]:
%sql
-- How many duplicates are in Bronze orders?
SELECT order_id, COUNT(*) AS cnt
FROM dev.dbx_course.bronze_orders
GROUP BY order_id
HAVING cnt > 1;

In [0]:
%sql
-- Silver orders: deduplicate, cast types, validate nulls, derive order_total
-- ROW_NUMBER deduplication: keep latest record per order_id
-- Null filter: any record missing a key business field is excluded from Silver

DROP TABLE IF EXISTS dev.dbx_course.silver_orders;

CREATE TABLE dev.dbx_course.silver_orders AS
SELECT
    CAST(order_id AS INT)                AS order_id,
    customer_id,
    product_id,
    CAST(quantity AS INT)                AS quantity,
    CAST(unit_price AS DECIMAL(10,2))    AS unit_price,
    CAST(order_date AS DATE)             AS order_date,
    status,
    CAST(quantity AS INT) * CAST(unit_price AS DECIMAL(10,2)) AS order_total, 
    --simple multiplication that belongs in Silver rather than being recomputed in every Gold query that needs it
    _ingest_timestamp,
    _source
FROM (
    SELECT *,
        ROW_NUMBER() OVER (
            PARTITION BY order_id
            ORDER BY _ingest_timestamp DESC
        ) AS rn
    FROM dev.dbx_course.bronze_orders
    WHERE order_id IS NOT NULL
      AND customer_id IS NOT NULL
      AND quantity IS NOT NULL
      AND unit_price IS NOT NULL
)
WHERE rn = 1;

In [0]:
%sql
-- 8 rows expected: 9 Bronze rows minus 1 duplicate
SELECT * FROM dev.dbx_course.silver_orders ORDER BY order_id;

In [0]:
%sql
-- Check customer duplicates
SELECT customer_id, COUNT(*) AS cnt
FROM dev.dbx_course.bronze_customers
GROUP BY customer_id
HAVING cnt > 1;

In [0]:
%sql
DROP TABLE IF EXISTS dev.dbx_course.silver_customers;

CREATE TABLE dev.dbx_course.silver_customers AS
SELECT
    customer_id,
    first_name,
    last_name,
    email,
    city,
    CAST(signup_date AS DATE) AS signup_date,
    tier,
    _ingest_timestamp,
    _source
FROM (
    SELECT *,
        ROW_NUMBER() OVER (
            PARTITION BY customer_id
            ORDER BY _ingest_timestamp DESC
        ) AS rn
    FROM dev.dbx_course.bronze_customers
    WHERE customer_id IS NOT NULL
      AND email IS NOT NULL
)
WHERE rn = 1;

In [0]:
%sql
-- 4 rows expected: 5 Bronze rows minus 1 duplicate
SELECT * FROM dev.dbx_course.silver_customers ORDER BY customer_id;

####Production Pattern: Incremental Silver

In [0]:
%sql
-- Incremental Silver load pattern: MERGE for idempotent daily runs
-- _ingest_timestamp filter ensures only new Bronze records are processed
    -- look at the highest timestamp already in Silver and only processes Bronze records that arrived after that point.
    -- This pattern scales — it works the same way whether Bronze has 10,000 records or 10 billion.
-- Run this on day 2, day 3, day N — result is always correct

MERGE INTO dev.dbx_course.silver_orders AS target
USING (
    SELECT
        CAST(order_id AS INT)                AS order_id,
        customer_id,
        product_id,
        CAST(quantity AS INT)                AS quantity,
        CAST(unit_price AS DECIMAL(10,2))    AS unit_price,
        CAST(order_date AS DATE)             AS order_date,
        status,
        CAST(quantity AS INT) * CAST(unit_price AS DECIMAL(10,2)) AS order_total,
        _ingest_timestamp,
        _source
    FROM (
        SELECT *,
            ROW_NUMBER() OVER (
                PARTITION BY order_id
                ORDER BY _ingest_timestamp DESC
            ) AS rn
        FROM dev.dbx_course.bronze_orders
        WHERE order_id IS NOT NULL
          AND _ingest_timestamp > (SELECT MAX(_ingest_timestamp) FROM dev.dbx_course.silver_orders)
    )
    WHERE rn = 1
) AS source
ON target.order_id = source.order_id
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *;

---
## Section 3: Gold Layer
**Design principle:** Answer a specific business question.\
Everything in Bronze and Silver was about correctness and preservation. Gold is about speed and usability.\
Gold tables are pre-aggregated, read-optimised, and built for a specific consumer — a BI tool, a dashboard, a weekly business report.

The design question you will face on every Gold table is: should this be a materialised table or a view? 
* A view is simpler — it just wraps the Silver query. 
* A materialised table stores the result. 

The answer depends on query frequency and computation cost. If a Gold metric is queried hundreds of times a day and the underlying Silver join is expensive, materialise it.\
If it is queried once a day and Silver is fast, a view is fine. 

We will build a materialised table here because that is the production-realistic choice for a revenue metric.

In [0]:
%sql
-- Gold: daily revenue by customer tier
-- Business rule: only completed orders count as revenue
-- This filter is a documented Gold decision, not a Silver contract

DROP TABLE IF EXISTS dev.dbx_course.gold_daily_revenue;

CREATE TABLE dev.dbx_course.gold_daily_revenue AS
SELECT
    o.order_date,
    c.tier                          AS customer_tier,
    COUNT(DISTINCT o.order_id)      AS order_count,
    COUNT(DISTINCT o.customer_id)   AS unique_customers,
    SUM(o.order_total)              AS total_revenue,
    ROUND(AVG(o.order_total), 2)    AS avg_order_value,
    CURRENT_TIMESTAMP()             AS last_refreshed
FROM dev.dbx_course.silver_orders o
JOIN dev.dbx_course.silver_customers c
    ON o.customer_id = c.customer_id
WHERE o.status = 'completed'
GROUP BY o.order_date, c.tier
ORDER BY o.order_date, c.tier;

In [0]:
%sql
SELECT * FROM dev.dbx_course.gold_daily_revenue;

Notice what we excluded: Anything WHERE status is not 'completed'. It is a business rule.\
This is appropriate in Gold — Gold is opinionated. It knows what the business counts as revenue.\
That filter does not belong in Silver because Silver should be complete.\
If someone later asks "what was the value of cancelled orders?" Silver can answer that.\
Gold cannot, and that is fine — Gold is not trying to answer every question, just the one it was built for.

In [0]:
%sql
-- Alternative: Gold as a view
-- Always current, no refresh required, re-runs the query on every call
-- Choose this when data freshness matters more than query speed

DROP VIEW IF EXISTS dev.dbx_course.gold_daily_revenue_view;

CREATE VIEW dev.dbx_course.gold_daily_revenue_view AS
SELECT
    o.order_date,
    c.tier                          AS customer_tier,
    COUNT(DISTINCT o.order_id)      AS order_count,
    COUNT(DISTINCT o.customer_id)   AS unique_customers,
    SUM(o.order_total)              AS total_revenue,
    ROUND(AVG(o.order_total), 2)    AS avg_order_value
FROM dev.dbx_course.silver_orders o
JOIN dev.dbx_course.silver_customers c
    ON o.customer_id = c.customer_id
WHERE o.status = 'completed'
GROUP BY o.order_date, c.tier;

In [0]:
%sql
SELECT * FROM dev.dbx_course.gold_daily_revenue_view ORDER BY order_date, customer_tier;